In [4]:
#Librerias
import pandas as pd
import json
import re

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Realizamos un  primer análisis para ver la composición de las colunas y de los datos en el archivo

In [ ]:
file_path = '/content/10.json'
with open(file_path, 'r') as f:
    data = [json.loads(line) for line in f]

df = pd.json_normalize(data)
print(df.head())

print(df.columns)

print("Tamaño del conjunto de datos:", df.shape)

               name                                            address  \
0   Porter Pharmacy  Porter Pharmacy, 129 N Second St, Cochran, GA ...   
1  Clean Air Vapors  Clean Air Vapors, 401 W Main St, Riverton, WY ...   
2          Boot Bar        Boot Bar, 702 E Main St, Riverton, WY 82501   
3        Boysen Dam                     Boysen Dam, Shoshoni, WY 82649   
4        Cafe Dacha  Cafe Dacha, 675 Central Ave Suite 1, Highland ...   

                                 gmap_id  \
0  0x88f16e41928ff687:0x883dad4fd048e8f8   
1  0x8758dd1117f560df:0xe691711d0c88da68   
2  0x8758dd1d120d3c0d:0xe49a6c2994c781c0   
3  0x8758af9fcdabbee7:0xefa71efcf509afe8   
4  0x880fc1dd4696911b:0x1952736e51ae8b7c   

                                         description   latitude   longitude  \
0                                               None  32.388300  -83.357100   
1                                               None  43.024439 -108.395705   
2  Retail chain with a variety of Western & work-... 

Separamos los datos del archivo que solo corresponden a restaurant en base alas cordenadas de Florida

In [ ]:
# Cargamos el archivo JSON reparado
archivo = '/content/10.json'
df = pd.read_json(archivo, lines=True)

# Escojemos el rango geográfico de Florida a clasificar
lat_min, lat_max = 24.396308, 31.000000
lon_min, lon_max = -87.634938, -79.974307

# Palabras clave relacionadas con restaurantes
keywords = ["Restaurant", "Café", "Coffee Shop", "Bar", "Food", "Diner", "Eatery"]

# Filtramos los datos de Florida y categorías relacionadas
florida_food_places = df[
    (df['latitude'] >= lat_min) & (df['latitude'] <= lat_max) &
    (df['longitude'] >= lon_min) & (df['longitude'] <= lon_max) &
    (df['category'].apply(
        lambda categories: any(keyword in categories for keyword in keywords) if isinstance(categories, list) else False
    ))
]

# Confirmamos la cantidad de lugares filtrados
print(f"Cantidad de lugares de comida en Florida: {len(florida_food_places)}")

# Guardamos los datos filtrados en un nuevo archivo JSON
if not florida_food_places.empty:
    florida_food_places.to_json("florida_food_places.json", orient="records", lines=True)
    print("Archivo 'florida_food_places.json' guardado con éxito.")
else:
    print("No se encontraron lugares de comida en Florida.")


Cantidad de lugares de comida en Florida: 1007
Archivo 'florida_food_places.json' guardado con éxito.


In [ ]:
file_path = '/content/11.json'
with open(file_path, 'r') as f:
    data = [json.loads(line) for line in f]

df = pd.json_normalize(data)
print(df.head())

print(df.columns)

print("Tamaño del conjunto de datos:", df.shape)

                                 name  \
0                     Porter Pharmacy   
1                      Mcguinns store   
2               Gentle Hands Grooming   
3  Smokecignals Electronic Cigarettes   
4     Advanced Home Medical Equipment   

                                             address  \
0  Porter Pharmacy, 129 N Second St, Cochran, GA ...   
1   Mcguinns store, 4884 NC-9, Mill Spring, NC 28756   
2  Gentle Hands Grooming, 156 US-176, Saluda, NC ...   
3  Smokecignals Electronic Cigarettes, 2108 W Tho...   
4  Advanced Home Medical Equipment, 885 Franklin ...   

                                 gmap_id description   latitude  longitude  \
0  0x88f16e41928ff687:0x883dad4fd048e8f8        None  32.388300 -83.357100   
1  0x88575f0654bd7c03:0xca3e467f7e766ad5        None  35.359437 -82.179693   
2  0x8859db7fe56d01c7:0xf90f3bdb3c62ba47        None  35.236050 -82.351049   
3  0x862722f489700e87:0x62c847c188bd5aa9        None  30.501326 -90.484117   
4  0x88523ca20145acc7:0x4b

Unificamos los tres archivos en uno solo

In [5]:
# Cargamos los archivos a combinar
archivos = [
    '/content/drive/MyDrive/data/datos en bruto/google/florida_food_places_combined.ipynb',
    '/content/drive/MyDrive/data/datos en bruto/google/florida_food_places_combined.json',
    '/content/drive/MyDrive/data/datos en bruto/google/florida_food_places_combined.json'
]

# Cargamos y combinamos todos los archivos
dataframes = [pd.read_json(archivo, lines=True) for archivo in archivos]
df_combined = pd.concat(dataframes, ignore_index=True)

# Convertimos listas en columnas a cadenas temporales
for col in df_combined.columns:
    if df_combined[col].apply(lambda x: isinstance(x, list)).any():
        df_combined[col] = df_combined[col].apply(lambda x: str(x) if isinstance(x, list) else x)



# Guardamos el archivo combinado en formato JSON
archivo_combinado = '/content/sample_data/florida_food_places_combined.json'
df_combined.to_json(archivo_combinado, orient='records', lines=True)

print(f"Archivos fusionados y guardados en '{archivo_combinado}'.")


Archivos fusionados y guardados en '/content/sample_data/florida_food_places_combined.json'.


Combertimos a formato csv

In [6]:
# Car4gamos la ruta del archivo JSON
file_path = "/content/sample_data/florida_food_places_combined.json"

# Leemos  el archivo JSON línea por línea
data = pd.read_json(file_path, lines=True)

# Filtramos las filas donde la columna 'category' contiene la palabra 'restaurant'
filtered_data = data[data['category'].str.contains('restaurant', case=False, na=False)]

# Guardamos los datos filtrados en un nuevo archivo  CSV

filtered_data.to_csv("/content/sample_data/filtered_restaurants.csv", index=False)

print("Datos filtrados guardados en JSON y CSV.")


Datos filtrados guardados en JSON y CSV.


Renombramos columnas

In [7]:
# Cargar el archivo
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Error: No se encontró el archivo en {file_path}")
    raise

# Renombramos columnas a usarcolumnas
column_mapping = {
    'name': 'nombre',
    'latitude': 'latitud',
    'longitude': 'longitud',
    'category': 'cocina',
    'avg_rating': 'puntuacion_usuarios'
}

df = df.rename(columns=column_mapping)

# Guardamos el archivo con los cambios
output_path = '/content/sample_data/processed_restaurants2_renamed.csv'
df.to_csv(output_path, index=False)

print(f"Archivo procesado y guardado en: {output_path}")


<ipython-input-7-d06dc95c3d60>:3: DtypeWarning: Columns (90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Archivo procesado y guardado en: /content/sample_data/processed_restaurants2_renamed.csv


Creamos las columnas id nombres id ciudad

In [ ]:
# Rutas de los archivos que usaremos
restaurants_file = '/content/sample_data/processed_restaurants2.csv'
names_file = '/content/sample_data/nombres_restaurant.csv'
cities_file = '/content/sample_data/ciudades_dim.csv'

# Cargamos los archivos
try:
    df_restaurants = pd.read_csv(restaurants_file)
    df_names = pd.read_csv(names_file)
    df_cities = pd.read_csv(cities_file)
except FileNotFoundError as e:
    print(f"Error: {e}")
    raise

# Verificamos si las columnas necesarias existen en el archivo
def validate_columns(df, required_columns):
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise KeyError(f"Faltan las siguientes columnas: {missing_columns}")

validate_columns(df_restaurants, ['nombre', 'ciudad'])
validate_columns(df_names, ['id_nombre', 'nombre'])
validate_columns(df_cities, ['id_ciudad', 'ciudad'])

# Creacreamos las Ids únicas para nombres de restaurantes
unique_names = df_restaurants['nombre'].unique()
existing_names = df_names['nombre'].tolist()

# Verificamos y agregamos nuevos nombres
new_names = [name for name in unique_names if name not in existing_names]
if new_names:
    new_name_ids = [{'id_nombre': hashlib.md5(name.encode()).hexdigest()[:4], 'nombre': name} for name in new_names]
    df_new_names = pd.DataFrame(new_name_ids)
    df_names = pd.concat([df_names, df_new_names], ignore_index=True)
    df_names.to_csv(names_file, index=False)

# Mapeamos las ids de nombres en df_restaurants
name_to_id_map = pd.Series(df_names['id_nombre'].values, index=df_names['nombre']).to_dict()
df_restaurants['id_nombre'] = df_restaurants['nombre'].map(name_to_id_map)

# Creamos las ids únicas para ciudades
unique_cities = df_restaurants['ciudad'].unique()
existing_cities = df_cities['ciudad'].tolist()

# Verificamos el el archivo ciudades_dim y agregamos nuevas ciudades de aver
new_cities = [city for city in unique_cities if city not in existing_cities]
if new_cities:
    new_city_ids = [{'id_ciudad': hashlib.md5(city.encode()).hexdigest()[:4], 'ciudad': city} for city in new_cities]
    df_new_cities = pd.DataFrame(new_city_ids)
    df_cities = pd.concat([df_cities, df_new_cities], ignore_index=True)
    df_cities.to_csv(cities_file, index=False)

# Mapeamos las ids de ciudades en df_restaurants
city_to_id_map = pd.Series(df_cities['id_ciudad'].values, index=df_cities['ciudad']).to_dict()
df_restaurants['id_ciudad'] = df_restaurants['ciudad'].map(city_to_id_map)

# Guardamos el resultado
output_file = '/content/sample_data/processed_restaurants_ids.csv'
df_restaurants.to_csv(output_file, index=False)

print(f"Archivo procesado y guardado en: {output_file}")


Archivo procesado y guardado en: /content/sample_data/processed_restaurants_ids.csv


Prosedemos a aser un merget con el archivo reviews_florida.csv  donde dejaremos la columna nombres y sus ids. Realizamos un analisis de sentimientos

In [ ]:
# Rutas de los archivos
restaurants_file = '/content/sample_data/filtered_restaurants.csv'
reviews_file = '/content/sample_data/reviews_florida.csv'
output_file = '/content/sample_data/merged_restaurant_reviews.csv'

# Cargamos los datos de restaurantes
restaurants_df = pd.read_csv(restaurants_file)

if 'nombres' not in restaurants_df.columns:
    restaurants_df['nombres'] = restaurants_df['name']

# Cargamos los datos de reseñas con manejo de tipos y low_memory desactivado
reviews_df = pd.read_csv(reviews_file, header=None, names=[
    'review_id', 'author', 'rating', 'review_text', 'owner_response', 'gmap_id'
], dtype={'review_id': str, 'author': str, 'review_text': str, 'owner_response': str, 'gmap_id': str}, low_memory=False)

# Convertimos la columna 'rating' a numérico, manejando errores
reviews_df['rating'] = pd.to_numeric(reviews_df['rating'], errors='coerce')

# Eliminamos las columnas innecesarias de reseñas
reviews_df = reviews_df.drop(columns=['owner_response'])

# Relacionamos  reseñas con restaurantes por la columna'gmap_id'
merged_df = pd.merge(reviews_df, restaurants_df, how='inner', left_on='gmap_id', right_on='gmap_id')

# Seleccionamos las columnas relevantes después de la unión
merged_df = merged_df[['nombres', 'review_text']]

# Realizamos un análisis de sentimientos alos comentarios
def sentiment_analysis(text):
    if pd.isna(text):
        return 0
    sentiment = TextBlob(text).sentiment.polarity
    if sentiment > 0:
        return 1
    elif sentiment < 0:
        return 0.5
    else:
        return 0

merged_df['analisis_sentimientos'] = merged_df['review_text'].apply(sentiment_analysis)

# Aproximamos los valores de análisis de sentimientos a 0, 0.5 o 1
merged_df['analisis_sentimientos'] = merged_df['analisis_sentimientos'].apply(lambda x: 1 if x > 0.75 else (0.5 if x > 0 and x <= 0.75 else 0))

# Renombramos la  columna 'nombres' a 'nombre'
merged_df = merged_df.rename(columns={'nombres': 'nombre'})

# Eliminamos la columna 'review_text'
merged_df = merged_df.drop(columns=['review_text'])

# Calculamos el promedio de análisis de sentimientos por restaurante y redondear
result_df = merged_df.groupby('nombre', as_index=False).mean()
result_df['analisis_sentimientos'] = result_df['analisis_sentimientos'].apply(lambda x: 1 if x > 0.75 else (0.5 if x > 0 and x <= 0.75 else 0))

# Guardamos el archivo final
final_output_file = '/content/sample_data/final_restaurant_sentiments.csv'
result_df.to_csv(final_output_file, index=False)

print(f"Archivo final guardado en: {final_output_file}")


Archivo final guardado en: /content/sample_data/final_restaurant_sentiments.csv


realisamos un merge entre el archivo final_restaurant_sentiments.csv y nombres_restaurant.csv



In [ ]:
# Ruta de los archivos
restaurants_file = '/content/sample_data/final_restaurant_sentiments.csv'
names_file = '/content/sample_data/nombres_restaurant.csv'

# Cargamos los archivos
try:
    df_restaurants = pd.read_csv(restaurants_file)
    df_names = pd.read_csv(names_file)
except FileNotFoundError as e:
    print(f"Error: {e}")
    raise
def validate_columns(df, required_columns):
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise KeyError(f"Faltan las siguientes columnas: {missing_columns}")

validate_columns(df_restaurants, ['nombre'])
validate_columns(df_names, ['id_nombre', 'nombre'])

# Mapeamos las ids de nombres en df_restaurants
name_to_id_map = pd.Series(df_names['id_nombre'].values, index=df_names['nombre']).to_dict()
df_restaurants['id_nombre'] = df_restaurants['nombre'].map(name_to_id_map)

# Guardamos el archivo con los cambios
output_path = '/content/sample_data/final_restaurant_sentiments_with_ids.csv'
df_restaurants.to_csv(output_path, index=False)

print(f"Archivo procesado y guardado en: {output_path}")


Archivo procesado y guardado en: /content/sample_data/final_restaurant_sentiments_with_ids.csv


In [ ]:
# Rutas de los archivos
sentiments_file = '/content/sample_data/final_restaurant_sentiments_with_ids.csv'
restaurants_file = '/content/sample_data/processed_restaurants_ids.csv'

try:
    df_sentiments = pd.read_csv(sentiments_file)
    df_restaurants = pd.read_csv(restaurants_file)
except FileNotFoundError as e:
    print(f"Error: {e}")
    raise

# Realizamos el join en la columna 'id_nombre'
df_merged = pd.merge(df_sentiments, df_restaurants, on='id_nombre', how='inner')

# Guardamos el archivo con los datos combinados
output_path = '/content/sample_data/merged_restaurant_data.csv'
df_merged.to_csv(output_path, index=False)

print(f"Archivo combinado y guardado en: {output_path}")


Archivo combinado y guardado en: /content/sample_data/merged_restaurant_data1.csv


Cambiamos nombres y reordenamos columnas

In [ ]:
# Eliminamos columnas innecesarias y renombramos las que bamos a usar
columns_to_drop = ['nombre_x', 'relative_results']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

df = df.rename(columns={
    'nombre_y': 'nombre',
    'cosina': 'categories',
    'avg_rating': 'puntuacion_usuarios',
    'sentiment_analysis': 'analisis_sentimientos'
})

# Convertimos la columna codigo_postal a entero
df['codigo_postal'] = pd.to_numeric(df['codigo_postal'], errors='coerce', downcast='integer')

# Reordenamos las columnas
column_order = [
    'id_nombre', 'nombre', 'direccion', 'id_ciudad', 'ciudad', 'estado',
    'codigo_postal', 'latitud', 'longitud', 'categories',
    'puntuacion_usuarios', 'analisis_sentimientos'
]

existing_columns = [col for col in column_order if col in df.columns]
df = df[existing_columns]

# Guardamos el archivo con los datos procesados
output_path = '/content/sample_data/restaurant 2024.csv'
df.to_csv(output_path, index=False)

print(f"Archivo procesado y guardado en: {output_path}")


Archivo procesado y guardado en: /content/sample_data/restaurant 2024.csv


combertimos el archivo a csv

In [ ]:
# Guardamos el archivo en formato Parquet
output_path = '/content/sample_data/restaurant_2024.parquet'
df.to_parquet(output_path, index=False)

print(f"Archivo convertido y guardado en formato Parquet en: {output_path}")


Archivo convertido y guardado en formato Parquet en: /content/sample_data/restaurant_2024.parquet
